In [ ]:
#@title Setup (run once, then collapse)
!pip install -q ipywidgets plotly
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import plotly.graph_objects as go
print('Ready!')

# Section 4: RAG - Building Knowledge-Grounded AI

## Coming From Section 3

You've mastered LLM fundamentals - tokens, temperature, prompts. But what happens when you need the LLM to answer questions about YOUR company's data? That's RAG - Retrieval-Augmented Generation - the pattern that grounds AI in your knowledge base.

---

## The Lawyer's $5,000 Mistake

In 2023, attorneys at a New York law firm used ChatGPT to research case precedents. They submitted a court brief citing **6 cases that didn't exist** - the LLM hallucinated them completely, with fake case names, fake citations, and fake rulings.

The judge was not amused. The lawyers were **fined $5,000 and publicly sanctioned**.

**The PM lesson:** An LLM without grounding in real documents will confidently make up facts. RAG solves this - but only if implemented correctly.

---

## Interactive Tools

| Tool | Link |
|------|------|
| RAG Playground | [Launch](https://huggingface.co/spaces/axelsirota/rag-playground) |
| RAG Failure Simulator | [Launch](https://huggingface.co/spaces/axelsirota/rag-failure-simulator) |
| Chunking Visualizer | [Launch](https://huggingface.co/spaces/axelsirota/chunking-visualizer) |
| RAG vs Fine-tuning | [Launch](https://huggingface.co/spaces/axelsirota/rag-vs-finetuning) |
| Vector DB Cost Calculator | [Launch](https://huggingface.co/spaces/axelsirota/vector-db-cost-calculator) |

---

## Exercise 1: RAG vs Fine-tuning Decision

Your engineering team proposes different approaches for grounding AI in company knowledge. For each scenario, decide: **RAG, Fine-tuning, or Long Context?**

**Quick Reference:**
- **RAG**: Retrieve relevant docs at query time, feed to LLM. Best for: frequently changing info, need citations, large doc corpus.
- **Fine-tuning**: Retrain model on your data. Best for: stable domain knowledge, specific style/format, high volume.
- **Long Context**: Stuff all docs in the prompt. Best for: small doc sets (<100K tokens), simple use cases.

In [ ]:
#@title Exercise 1: RAG vs Fine-tuning Decision

scenarios = [
    {
        "title": "Customer Support Bot - Product FAQ",
        "description": "10,000 support articles that get updated weekly. Customers ask questions, need accurate answers with links to source articles.",
        "correct": "RAG",
        "explanation": "RAG is ideal here: frequent updates (weekly), large corpus (10K articles), need for citations. Fine-tuning would require constant retraining. Long context can't fit 10K articles."
    },
    {
        "title": "Medical Note Summarizer",
        "description": "Summarize doctor's notes into structured formats. Notes follow specific medical terminology. Format must be consistent. 50,000 notes/day.",
        "correct": "Fine-tuning",
        "explanation": "Fine-tuning wins: stable format requirements, domain-specific terminology, high volume (cost matters), no need for citations. RAG would add latency and cost without benefit."
    },
    {
        "title": "Internal Policy Chatbot",
        "description": "Answer employee questions about 15 HR policy documents (total ~50 pages). Policies change quarterly.",
        "correct": "Long Context",
        "explanation": "Long context works: small corpus (~50 pages fits in 100K context), simple Q&A, quarterly updates are manageable. RAG adds unnecessary complexity for this size."
    },
    {
        "title": "Legal Contract Analyzer",
        "description": "Analyze contracts against company standards and relevant case law. Must cite specific clauses and precedents. 500+ contracts, 1000+ case summaries.",
        "correct": "RAG",
        "explanation": "RAG is essential: need citations (legal requirement), large corpus, can't fit everything in context. Fine-tuning can't provide citations or handle the volume of reference material."
    },
    {
        "title": "Brand Voice Email Writer",
        "description": "Generate marketing emails in your brand's specific tone. Have 200 examples of approved emails. Need consistent voice across all outputs.",
        "correct": "Fine-tuning",
        "explanation": "Fine-tuning fits: stable style requirements, have training examples, no need for retrieval or citations. Few-shot prompting might work too, but fine-tuning ensures consistency at scale."
    }
]

rag_dds = []
output1 = widgets.Output()

for i, s in enumerate(scenarios):
    display(HTML(f"<h4>{i+1}. {s['title']}</h4><p>{s['description']}</p>"))
    dd = widgets.Dropdown(
        options=['-- Select --', 'RAG', 'Fine-tuning', 'Long Context'],
        value='-- Select --',
        description='Approach:'
    )
    rag_dds.append(dd)
    display(dd)

def check_rag(btn):
    output1.clear_output()
    score = 0
    with output1:
        for dd, s in zip(rag_dds, scenarios):
            correct = dd.value == s['correct']
            if correct:
                score += 1
            icon = '\u2705' if correct else '\u274c'
            display(HTML(f"<p>{icon} <b>{s['title']}</b>: Best approach: <b>{s['correct']}</b>. {s['explanation']}</p>"))
        pct = score / len(scenarios) * 100
        display(HTML(f"<h3>Score: {score}/{len(scenarios)} ({pct:.0f}%)</h3>"))
        if pct < 80:
            display(HTML("<p><em>Tip: Think about update frequency, corpus size, and whether you need citations.</em></p>"))

btn1 = widgets.Button(description='Check Answers', button_style='primary')
btn1.on_click(check_rag)
display(btn1, output1)

## Exercise 2: Chunking Impact Simulator

**The PM Question:** Your engineering team says they'll "chunk the documents." What does that mean, and why should you care?

**Chunking** splits your documents into smaller pieces for embedding and retrieval. Get it wrong, and your RAG system retrieves irrelevant or incomplete information.

Adjust the sliders below to see how chunk size and overlap affect your document.

In [ ]:
#@title Exercise 2: Chunking Impact Simulator

sample_doc = """Our return policy allows customers to return most items within 30 days of purchase for a full refund. Items must be in original condition with tags attached. Electronics have a 15-day return window due to rapid depreciation. Opened software and digital downloads are non-refundable. Gift cards cannot be returned or exchanged for cash. For items purchased during sales events, the return window starts from the original purchase date, not the sale date. Defective items can be returned within 90 days regardless of condition. International orders may have different return policies based on local regulations. Shipping costs for returns are the customer's responsibility unless the item is defective or we made an error. Refunds are processed within 5-7 business days after we receive the returned item."""

chunk_size = widgets.IntSlider(value=200, min=50, max=500, step=25, description='Chunk Size:', style={'description_width': '100px'})
overlap = widgets.IntSlider(value=20, min=0, max=50, step=5, description='Overlap %:', style={'description_width': '100px'})
output2 = widgets.Output()

def update_chunks(change):
    output2.clear_output()
    with output2:
        size = chunk_size.value
        ovlp = int(size * overlap.value / 100)
        step = max(size - ovlp, 1)
        
        chunks = []
        i = 0
        while i < len(sample_doc):
            chunk = sample_doc[i:i+size]
            chunks.append(chunk)
            i += step
        
        # Analysis
        avg_len = sum(len(c) for c in chunks) / len(chunks) if chunks else 0
        
        # Check for split sentences
        split_count = 0
        for c in chunks[:-1]:  # Skip last chunk
            if not c.rstrip().endswith(('.', '!', '?')):
                split_count += 1
        
        # Display stats
        display(HTML(f"""
        <div style='background:#f0f9ff; padding:15px; border-radius:8px; margin-bottom:15px;'>
            <h4 style='margin-top:0;'>Chunking Results</h4>
            <p><b>Total chunks:</b> {len(chunks)}</p>
            <p><b>Average chunk size:</b> {avg_len:.0f} characters</p>
            <p><b>Chunks with split sentences:</b> {split_count} {'⚠️' if split_count > 0 else '✅'}</p>
        </div>
        """))
        
        # Plotly chart
        fig = go.Figure(data=[
            go.Bar(
                x=[f'Chunk {i+1}' for i in range(len(chunks))],
                y=[len(c) for c in chunks],
                marker_color=['#dc2626' if not chunks[i].rstrip().endswith(('.', '!', '?')) and i < len(chunks)-1 else '#40B8A6' for i in range(len(chunks))]
            )
        ])
        fig.update_layout(
            title='Chunk Sizes (Red = Split Mid-Sentence)',
            xaxis_title='Chunk',
            yaxis_title='Characters',
            height=300,
            margin=dict(l=40, r=40, t=60, b=40)
        )
        fig.show()
        
        # Show first 3 chunks
        display(HTML("<h4>Preview: First 3 Chunks</h4>"))
        for i, c in enumerate(chunks[:3]):
            border_color = '#dc2626' if not c.rstrip().endswith(('.', '!', '?')) and i < len(chunks)-1 else '#40B8A6'
            display(HTML(f"""
            <div style='border:2px solid {border_color}; padding:10px; margin:5px 0; border-radius:6px; font-size:0.9em;'>
                <b>Chunk {i+1}:</b> {c}
            </div>
            """))
        
        # PM insight
        if split_count > len(chunks) * 0.3:
            display(HTML("""
            <div style='background:#fef2f2; padding:10px; border-radius:6px; margin-top:15px;'>
                <b>⚠️ PM Alert:</b> More than 30% of chunks split mid-sentence. This can cause retrieval to return incomplete information. Ask engineering about sentence-aware chunking.
            </div>
            """))
        elif overlap.value < 10:
            display(HTML("""
            <div style='background:#fefce8; padding:10px; border-radius:6px; margin-top:15px;'>
                <b>⚠️ Low Overlap:</b> With less than 10% overlap, context at chunk boundaries may be lost. Consider 10-20% overlap for better retrieval.
            </div>
            """))
        else:
            display(HTML("""
            <div style='background:#f0fdf4; padding:10px; border-radius:6px; margin-top:15px;'>
                <b>✅ Looks reasonable.</b> But always test with real queries to verify retrieval quality.
            </div>
            """))

chunk_size.observe(update_chunks, names='value')
overlap.observe(update_chunks, names='value')

display(HTML("<p><b>Sample Document:</b> A 600-character return policy document.</p>"))
display(chunk_size, overlap)
update_chunks(None)

## Exercise 3: RAG Failure Mode Identification

RAG systems fail in predictable ways. As a PM, you need to recognize these failure modes so you can:
1. Ask the right questions during development
2. Diagnose issues when they occur
3. Communicate problems to stakeholders

**The 5 RAG Failure Modes:**
1. **Bad Chunking** - Important info split across chunks
2. **Wrong Retrieval** - Similar words, wrong context
3. **Hallucination Despite Context** - Model ignores retrieved docs
4. **Outdated Information** - Doc says X, model says Y
5. **Missing Information** - Question not covered in docs

In [ ]:
#@title Exercise 3: Identify the Failure Mode

failure_scenarios = [
    {
        "title": "The Price Quote Problem",
        "scenario": "User asks: 'What's the price of the Pro plan?' RAG retrieves a chunk about 'Pro features' and 'plan pricing' but the retrieved chunk ends mid-sentence: '...the Pro plan costs $99 per month, which includes' - and the rest is in the next chunk.",
        "correct": "Bad Chunking",
        "explanation": "The sentence about pricing was split between chunks. The model only sees partial information. Fix: Use sentence-aware or semantic chunking."
    },
    {
        "title": "The Wrong Department",
        "scenario": "User asks: 'What's the refund policy for software?' RAG retrieves chunks about 'software licensing' and 'refund processing times' - both mention 'software' and 'refund' but neither covers software-specific refund rules.",
        "correct": "Wrong Retrieval",
        "explanation": "Semantic similarity matched keywords but missed the actual policy. The chunks are related but don't answer the question. Fix: Improve chunking, add metadata filtering, or use hybrid search."
    },
    {
        "title": "The Confident Wrong Answer",
        "scenario": "Retrieved chunks clearly state: 'Returns must be made within 30 days.' The model responds: 'You can return items within 60 days for a full refund.' The 60-day policy exists nowhere in the documents.",
        "correct": "Hallucination Despite Context",
        "explanation": "The model ignored the retrieved context and hallucinated. This can happen with weaker models or when the prompt doesn't emphasize grounding. Fix: Stronger grounding prompts, better models, or output validation."
    },
    {
        "title": "The Stale Data Issue",
        "scenario": "Your docs say 'Shipping is free on orders over $50' (updated last month). The model says 'Shipping is free on orders over $35' - which was true 6 months ago and is in GPT-4's training data.",
        "correct": "Outdated Information",
        "explanation": "The model's training data conflicts with your current docs. The model may prefer its 'knowledge' over retrieved context. Fix: Explicit prompting to prioritize retrieved docs, or use a model with less domain knowledge."
    },
    {
        "title": "The Missing Answer",
        "scenario": "User asks: 'Do you offer student discounts?' Your docs don't mention student discounts at all. The model responds: 'Yes, we offer a 15% student discount with valid ID.'",
        "correct": "Missing Information",
        "explanation": "The question isn't covered in your docs. Instead of saying 'I don't know,' the model hallucinated an answer. Fix: Train the model to say 'I don't have information about that' when retrieval confidence is low."
    }
]

fail_dds = []
output3 = widgets.Output()

for i, s in enumerate(failure_scenarios):
    display(HTML(f"<h4>{i+1}. {s['title']}</h4><p style='background:#f8fafc; padding:10px; border-radius:6px;'>{s['scenario']}</p>"))
    dd = widgets.Dropdown(
        options=['-- Select --', 'Bad Chunking', 'Wrong Retrieval', 'Hallucination Despite Context', 'Outdated Information', 'Missing Information'],
        value='-- Select --',
        description='Failure Mode:'
    )
    fail_dds.append(dd)
    display(dd)

def check_failures(btn):
    output3.clear_output()
    score = 0
    with output3:
        for dd, s in zip(fail_dds, failure_scenarios):
            correct = dd.value == s['correct']
            if correct:
                score += 1
            icon = '\u2705' if correct else '\u274c'
            display(HTML(f"<p>{icon} <b>{s['title']}</b>: {s['correct']}. {s['explanation']}</p>"))
        pct = score / len(failure_scenarios) * 100
        display(HTML(f"<h3>Score: {score}/{len(failure_scenarios)} ({pct:.0f}%)</h3>"))

btn3 = widgets.Button(description='Check Answers', button_style='primary')
btn3.on_click(check_failures)
display(btn3, output3)

## Exercise 4: RAG Cost Calculator

RAG systems have multiple cost components. Use this calculator to estimate monthly costs for a RAG deployment.

**Cost Components:**
- **Embedding costs** - Converting docs and queries to vectors
- **Vector DB storage** - Storing embeddings
- **Vector DB queries** - Searching for similar vectors
- **LLM generation** - Generating answers from retrieved context

In [ ]:
#@title Exercise 4: RAG Cost Calculator

# Inputs
num_docs = widgets.IntSlider(value=10000, min=1000, max=100000, step=1000, description='Documents:', style={'description_width': '120px'})
avg_doc_size = widgets.IntSlider(value=2000, min=500, max=10000, step=500, description='Avg Doc Tokens:', style={'description_width': '120px'})
queries_per_day = widgets.IntSlider(value=1000, min=100, max=50000, step=100, description='Queries/Day:', style={'description_width': '120px'})
chunks_retrieved = widgets.IntSlider(value=5, min=1, max=20, step=1, description='Chunks/Query:', style={'description_width': '120px'})

llm_model = widgets.Dropdown(
    options=[
        ('GPT-4o ($2.50/$10 per 1M)', 'gpt4o'),
        ('GPT-4o-mini ($0.15/$0.60 per 1M)', 'gpt4omini'),
        ('Claude 3.5 Sonnet ($3/$15 per 1M)', 'sonnet'),
        ('Claude 3.5 Haiku ($0.80/$4 per 1M)', 'haiku')
    ],
    value='gpt4omini',
    description='LLM Model:',
    style={'description_width': '120px'}
)

output4 = widgets.Output()

# Pricing (per 1M tokens or per unit)
PRICING = {
    'embedding': 0.02,  # per 1M tokens (OpenAI ada-002)
    'vector_storage': 0.25,  # per GB/month (Pinecone)
    'vector_query': 0.0001,  # per query (estimated)
    'gpt4o': {'input': 2.50, 'output': 10.00},
    'gpt4omini': {'input': 0.15, 'output': 0.60},
    'sonnet': {'input': 3.00, 'output': 15.00},
    'haiku': {'input': 0.80, 'output': 4.00}
}

def calculate_costs(change):
    output4.clear_output()
    with output4:
        # Calculations
        total_tokens = num_docs.value * avg_doc_size.value
        chunks_per_doc = avg_doc_size.value / 500  # ~500 tokens per chunk
        total_chunks = num_docs.value * chunks_per_doc
        
        # One-time embedding cost (docs)
        embedding_cost_docs = (total_tokens / 1_000_000) * PRICING['embedding']
        
        # Monthly embedding cost (queries)
        query_tokens_month = queries_per_day.value * 30 * 100  # ~100 tokens per query
        embedding_cost_queries = (query_tokens_month / 1_000_000) * PRICING['embedding']
        
        # Vector storage (1536 dims * 4 bytes * chunks)
        storage_gb = (total_chunks * 1536 * 4) / (1024**3)
        storage_cost = storage_gb * PRICING['vector_storage']
        
        # Vector query cost
        query_cost = queries_per_day.value * 30 * PRICING['vector_query']
        
        # LLM cost
        model = llm_model.value
        context_tokens = chunks_retrieved.value * 500  # tokens per retrieved chunk
        input_tokens_month = queries_per_day.value * 30 * (100 + context_tokens)  # query + context
        output_tokens_month = queries_per_day.value * 30 * 300  # ~300 token response
        
        llm_input_cost = (input_tokens_month / 1_000_000) * PRICING[model]['input']
        llm_output_cost = (output_tokens_month / 1_000_000) * PRICING[model]['output']
        llm_cost = llm_input_cost + llm_output_cost
        
        # Total monthly (excluding one-time embedding)
        monthly_total = embedding_cost_queries + storage_cost + query_cost + llm_cost
        
        # Display results
        display(HTML(f"""
        <div style='background:#f0f9ff; padding:20px; border-radius:8px; margin-bottom:20px;'>
            <h3 style='margin-top:0; color:#1e40af;'>Monthly Cost Breakdown</h3>
            <table style='width:100%; border-collapse:collapse;'>
                <tr><td style='padding:8px; border-bottom:1px solid #e5e7eb;'>Embedding (queries)</td><td style='text-align:right; padding:8px; border-bottom:1px solid #e5e7eb;'>${embedding_cost_queries:.2f}</td></tr>
                <tr><td style='padding:8px; border-bottom:1px solid #e5e7eb;'>Vector DB Storage</td><td style='text-align:right; padding:8px; border-bottom:1px solid #e5e7eb;'>${storage_cost:.2f}</td></tr>
                <tr><td style='padding:8px; border-bottom:1px solid #e5e7eb;'>Vector DB Queries</td><td style='text-align:right; padding:8px; border-bottom:1px solid #e5e7eb;'>${query_cost:.2f}</td></tr>
                <tr><td style='padding:8px; border-bottom:1px solid #e5e7eb;'>LLM Generation</td><td style='text-align:right; padding:8px; border-bottom:1px solid #e5e7eb;'>${llm_cost:.2f}</td></tr>
                <tr style='font-weight:bold; background:#dbeafe;'><td style='padding:8px;'>MONTHLY TOTAL</td><td style='text-align:right; padding:8px;'>${monthly_total:.2f}</td></tr>
            </table>
            <p style='margin-top:15px; font-size:0.9em; color:#6b7280;'>One-time doc embedding cost: ${embedding_cost_docs:.2f}</p>
        </div>
        """))
        
        # Pie chart
        costs = [embedding_cost_queries, storage_cost, query_cost, llm_cost]
        labels = ['Embedding', 'Storage', 'Query', 'LLM']
        colors = ['#40B8A6', '#1A3D4D', '#6366f1', '#f59e0b']
        
        fig = go.Figure(data=[go.Pie(
            labels=labels,
            values=costs,
            marker_colors=colors,
            hole=0.4
        )])
        fig.update_layout(
            title='Cost Distribution',
            height=350,
            margin=dict(l=40, r=40, t=60, b=40)
        )
        fig.show()
        
        # Scaling warning
        if monthly_total > 1000:
            display(HTML("""
            <div style='background:#fef2f2; padding:10px; border-radius:6px; margin-top:15px;'>
                <b>⚠️ Cost Alert:</b> Monthly cost exceeds $1,000. Consider: smaller models for simple queries, caching frequent questions, or reducing chunks retrieved.
            </div>
            """))
        
        # LLM dominance warning
        if llm_cost > monthly_total * 0.8:
            display(HTML("""
            <div style='background:#fefce8; padding:10px; border-radius:6px; margin-top:15px;'>
                <b>💡 Insight:</b> LLM costs are 80%+ of total. The vector DB is cheap - your optimization efforts should focus on the LLM (model selection, prompt efficiency, caching).
            </div>
            """))

# Attach observers
for w in [num_docs, avg_doc_size, queries_per_day, chunks_retrieved, llm_model]:
    w.observe(calculate_costs, names='value')

display(HTML("<h4>Configure Your RAG System</h4>"))
display(num_docs, avg_doc_size, queries_per_day, chunks_retrieved, llm_model)
calculate_costs(None)

## Discussion Prompts

1. **Your engineering team proposes building a RAG system for customer support.** What questions do you ask before approving the project? Think about: data quality, update frequency, success metrics, failure handling.

2. **The RAG chatbot is giving wrong answers 15% of the time.** How do you diagnose whether it's a chunking problem, retrieval problem, or generation problem? What would you ask the team to investigate first?

3. **Your CFO asks: "Why do we need a vector database? We already have a database."** How do you explain the difference in a 30-second elevator pitch?

4. **A competitor launches a similar RAG-powered feature.** Your VP wants to know why yours isn't live yet. How do you frame the value of getting RAG right vs. shipping fast?

---

## Stakeholder Framing

### For Executives
> "RAG lets us give AI accurate answers grounded in our actual documents - product manuals, policies, FAQs - instead of relying on the AI's general knowledge. The lawyer who submitted fake case citations? That's what happens without RAG. The cost is primarily in the LLM, not the vector database."

### For Engineering
> "I need to understand: What chunking strategy are we using? How are we handling cases where the answer spans multiple chunks? What's our plan when retrieval confidence is low - do we say 'I don't know' or escalate to a human?"

### For Legal/Compliance
> "RAG gives us citations - we can show exactly which document the AI used to generate each answer. This is critical for audit trails. We should also discuss: how do we handle questions not covered in our docs? The AI should refuse to answer rather than guess."

---

## Key Takeaways

- **RAG grounds AI in your documents** - Prevents hallucination by retrieving real content before generating
- **Chunking matters more than you think** - Bad chunks = bad retrieval = wrong answers
- **Know the 5 failure modes** - Bad chunking, wrong retrieval, hallucination despite context, outdated info, missing info
- **LLM costs dominate** - Vector DB is cheap; optimize the LLM (model selection, caching, prompt efficiency)
- **RAG vs Fine-tuning is a PM decision** - Consider update frequency, citation needs, and corpus size
- **Always test with real queries** - Synthetic tests miss the edge cases users will find

---

## Up Next: Section 5 - Agents

RAG retrieves knowledge. But what if you need AI to **take actions** - send emails, update databases, make purchases? Section 5 covers AI Agents and intelligent automation.